# 12.8 - Monitoring & Observability

**Module 12 - Production Deployment** | Netsetos GenAI Engineering

This notebook works through Monitoring & Observability hands-on: Why AI observability is different; The three pillars (and a fourth); Traces and spans; OpenTelemetry and the gen_ai conventions; The golden signals; Alerting on SLOs and anomalies.

Read top to bottom: each code cell has a short **intro above** it and a **step-by-step walkthrough below** it. Run the cell, then check its output against the walkthrough.

## Setup

The lesson runs entirely on plain Python with seeded, hard-coded values - no API keys and no observability backend needed. This first cell just notes the one optional install and sets the stage for reproducible output.

In [ ]:
# Install dependencies (if running in Colab or fresh environment)
# Uncomment the next line if needed:
# !pip install anthropic -q

# Reproducibility - the lesson uses seeded random values throughout

**Explanation**

Environment prep only. The `anthropic` install is commented out because none of the runnable cells actually call an API - they model observability behavior with in-code data - and the comment flags that every number in the notebook is fixed rather than random.

**How the code works - step by step**
- **`# !pip install anthropic -q`** - commented-out install, uncomment only if you later run the illustrative Anthropic call.
- **Reproducibility note** - a reminder that the lesson uses seeded/fixed values throughout, so outputs are deterministic.

**In one line:** nothing executes here - it just prepares a keyless, deterministic environment.

**What you'll see:** (no output - environment prep)

## 1 - Why AI observability is different

A normal web app fails in three ways an ops dashboard already watches: it goes down, it gets slow, or it throws errors. An AI app has a fourth failure mode those dashboards have no gauge for - it gets *worse*. This cell shows one model swap through both lenses so you can see the ops panel stay green while the things users feel quietly rot.

In [ ]:
# The ops dashboard watches a NORMAL web app: is it up, fast, erroring?
# An AI app can be perfect on all three while the things users feel quietly rot.
BEFORE = {"p95_ms": 2100, "error_rate": 0.000, "http_ok": 1.00, "quality": 8.4, "cost_per_reply": 0.011, "ttft_ms": 700}
AFTER  = {"p95_ms": 2160, "error_rate": 0.000, "http_ok": 1.00, "quality": 6.9, "cost_per_reply": 0.019, "ttft_ms": 2100}

def health(v, ok):
    return "green" if ok else "RED"

print("A model swap shipped. What each dashboard sees:")
print()
print("  OPS dashboard (built for a normal web app):")
print("    p95 latency   {:>5}ms -> {:>5}ms   {}".format(BEFORE["p95_ms"], AFTER["p95_ms"], health(0, True)))
print("    error rate    {:>5.0%} -> {:>5.0%}   {}".format(BEFORE["error_rate"], AFTER["error_rate"], health(0, True)))
print("    HTTP 200 rate {:>5.0%} -> {:>5.0%}   {}".format(BEFORE["http_ok"], AFTER["http_ok"], health(0, True)))
print("    => all green. On-call sees nothing.")
print()
print("  AI signals (what the users actually feel):")
print("    answer quality {:>4.1f} -> {:>4.1f}   {}".format(BEFORE["quality"], AFTER["quality"], health(0, False)))
print("    cost / reply  ${:>5.3f} -> ${:>5.3f}  {}".format(BEFORE["cost_per_reply"], AFTER["cost_per_reply"], health(0, False)))
print("    time-to-first-token {:>4}ms -> {:>4}ms  {}".format(BEFORE["ttft_ms"], AFTER["ttft_ms"], health(0, False)))
print("    => quality fell, cost nearly doubled, first word takes 3x longer.")
print()
print("Monitoring answers 'is it broken?'. Observability answers 'why does it feel worse?'.")
print("An AI app needs the second kind: instrument quality, cost, and TTFT, not just up/fast/errors.")

# Output:
# A model swap shipped. What each dashboard sees:
#
#   OPS dashboard (built for a normal web app):
#     p95 latency    2100ms ->  2160ms   green
#     error rate       0% ->    0%   green
#     HTTP 200 rate  100% ->  100%   green
#     => all green. On-call sees nothing.
#
#   AI signals (what the users actually feel):
#     answer quality  8.4 ->  6.9   RED
#     cost / reply  $0.011 -> $0.019  RED
#     time-to-first-token  700ms -> 2100ms  RED
#     => quality fell, cost nearly doubled, first word takes 3x longer.
#
# Monitoring answers 'is it broken?'. Observability answers 'why does it feel worse?'.
# An AI app needs the second kind: instrument quality, cost, and TTFT, not just up/fast/errors.

**Explanation**

A side-by-side illustration, not a real measurement - it hard-codes a BEFORE and AFTER snapshot of the same deploy and prints each through two dashboards. The point is contrast: three ops signals barely move (all green) while three AI-specific signals collapse, which is why an AI app needs observability, not just monitoring.

**How the code works - step by step**
- **`BEFORE` / `AFTER`** - two dicts holding the same six metrics before and after a quiet model swap.
- **`health(v, ok)`** - a trivial helper returning `"green"` or `"RED"` from a boolean, used to label each row.
- **Ops block** - prints p95 latency, error rate, and HTTP-200 rate, all forced green because they scarcely changed.
- **AI block** - prints answer quality, cost per reply, and time-to-first-token, all forced RED because they regressed hard.
- **Closing lines** - contrast monitoring ("is it broken?") with observability ("why does it feel worse?").

**In one line:** same deploy, two dashboards - ops sees nothing, the AI signals see the real regression.

**What you'll see:** Two labeled dashboards: the ops panel shows p95 2100->2160ms, error 0%, HTTP 100% all "green"; the AI panel shows quality 8.4->6.9, cost $0.011->$0.019, TTFT 700->2100ms all "RED", closing with the monitoring-vs-observability framing.

## 2 - The three pillars (and a fourth)

Logs, metrics, and traces each answer a different question, and reaching for the wrong one leaves you staring at data that cannot help. This cell takes a single slow request and shows it through all three pillars so the division of labor is concrete - plus a nod to the fourth pillar, evals.

In [ ]:
# One slow request, seen through the three pillars - each answers a DIFFERENT question.
# A LOG is one event with fields. A METRIC is a number over time. A TRACE is the causal chain.
request = {
    "id": "req-42", "route": "/chat", "user": "u_88", "status": 200,
    "model": "claude-sonnet-4-6", "latency_ms": 6100, "input_tokens": 1800, "output_tokens": 240,
}
p95_series = [2000, 2100, 2050, 2200, 6100]   # last 5 minutes of p95, this request is the spike
spans = [("retrieve_context", 200), ("llm_generate", 5800), ("format_reply", 40)]

print("PILLAR 1 - LOG  (question: what exactly happened in THIS event?)")
for k, v in request.items():
    print("    {:<14} {}".format(k, v))
print()
print("PILLAR 2 - METRIC  (question: how is p95 latency TRENDING?)")
print("    p95 over the last 5 minutes: " + " -> ".join("{}ms".format(x) for x in p95_series))
print("    the trend shows a spike; it does NOT tell you which request or why.")
print()
print("PILLAR 3 - TRACE  (question: WHERE did this one request spend its time?)")
total = sum(ms for _, ms in spans)
for name, ms in spans:
    print("    {:<16} {:>5}ms  ({:>2.0f}% of the request)".format(name, ms, 100 * ms / total))
print("    the trace pins the cost to llm_generate - the log and metric could not.")
print()
print("Logs, metrics, traces answer different questions; evals add a fourth: was the ANSWER any good?")

# Output:
# PILLAR 1 - LOG  (question: what exactly happened in THIS event?)
#     id             req-42
#     route          /chat
#     user           u_88
#     status         200
#     model          claude-sonnet-4-6
#     latency_ms     6100
#     input_tokens   1800
#     output_tokens  240
#
# PILLAR 2 - METRIC  (question: how is p95 latency TRENDING?)
#     p95 over the last 5 minutes: 2000ms -> 2100ms -> 2050ms -> 2200ms -> 6100ms
#     the trend shows a spike; it does NOT tell you which request or why.
#
# PILLAR 3 - TRACE  (question: WHERE did this one request spend its time?)
#     retrieve_context   200ms  ( 3% of the request)
#     llm_generate      5800ms  (96% of the request)
#     format_reply        40ms  ( 1% of the request)
#     the trace pins the cost to llm_generate - the log and metric could not.
#
# Logs, metrics, traces answer different questions; evals add a fourth: was the ANSWER any good?

**Explanation**

One request, three views - a teaching harness that renders the same event as a log, a metric series, and a span tree. The core idea is that each pillar answers a distinct question: the log gives an event's fields, the metric gives a trend, and only the trace localizes *where* the time went.

**How the code works - step by step**
- **`request`** - one event as a flat dict (id, route, user, status, model, latency, tokens): the LOG view.
- **`p95_series`** - five minutes of p95 values ending in the spike: the METRIC view, showing a trend but not a cause.
- **`spans`** - a list of (name, ms) pairs for the same request: the TRACE view.
- **Pillar 1 print** - dumps every field of `request`.
- **Pillar 2 print** - joins `p95_series` into a trend line and notes it can't say which request spiked.
- **Pillar 3 print** - sums the spans and prints each as a share of total, pinning the cost to `llm_generate`.

**In one line:** log = fields of one event, metric = a trend, trace = where the time went - three questions, three pillars.

**What you'll see:** Three labeled sections for one request: the log lists eight fields; the metric prints `2000ms -> 2100ms -> 2050ms -> 2200ms -> 6100ms`; the trace shows retrieve 3%, llm_generate 96%, format_reply 1%, ending on the reminder that evals are the fourth pillar.

## 3 - Traces and spans

The trace is the pillar that earns its keep in an AI app, so it gets its own cell. A trace is a tree of timed spans; drawn as a waterfall it shows, at a glance, where a request spent its time - and for an LLM app the answer is almost always the model-generation span.

In [ ]:
# A TRACE is a tree of SPANS. Each span is one unit of work with a start, a duration, and a parent.
# Rebuild the waterfall for one request and find where the time went.
# (name, parent, start_ms, duration_ms)
spans = [
    ("chat_request",   None,           0, 6130),
    ("retrieve",       "chat_request", 20, 200),
    ("llm_generate",   "chat_request", 230, 5800),
    ("tool_call",      "chat_request", 6030, 60),
    ("format_reply",   "chat_request", 6090, 40),
]
TTFT_MS = 800   # first streamed token arrived 800ms into llm_generate

root_dur = spans[0][3]
scale = 48.0 / root_dur   # 48 chars = the whole request
print("Trace for chat_request (one user message), total {}ms:".format(root_dur))
print()
for name, parent, start, dur in spans:
    depth = 0 if parent is None else 1
    pad = "  " * depth
    lead = int(start * scale)
    bar = "#" * max(1, int(dur * scale))
    pct = 100 * dur / root_dur
    print("  {}{:<14}{} {:<50} {:>5}ms  {:>2.0f}%".format(pad, name, "", " " * lead + bar, dur, pct))
print()
llm_start = next(s for n, p, s, d in spans if n == "llm_generate")
llm = next(d for n, p, s, d in spans if n == "llm_generate")
print("Reading the waterfall:")
print("  llm_generate is {:.0f}% of the request - the model call dominates, not your code.".format(100 * llm / root_dur))
print("  TTFT (first-token latency) was {}ms, measured inside the model span.".format(TTFT_MS))
print("  Blank-screen wait = {}ms to reach the model + {}ms TTFT = {}ms before ANY text appeared.".format(llm_start, TTFT_MS, llm_start + TTFT_MS))
print("  A single end-to-end latency number hid all of this; the span tree shows exactly where to optimize.")

# Output:
# Trace for chat_request (one user message), total 6130ms:
#
#   chat_request   ################################################    6130ms  100%
#     retrieve       #                                                    200ms   3%
#     llm_generate    #############################################      5800ms  95%
#     tool_call                                                     #      60ms   1%
#     format_reply                                                  #      40ms   1%
#
# Reading the waterfall:
#   llm_generate is 95% of the request - the model call dominates, not your code.
#   TTFT (first-token latency) was 800ms, measured inside the model span.
#   Blank-screen wait = 230ms to reach the model + 800ms TTFT = 1030ms before ANY text appeared.
#   A single end-to-end latency number hid all of this; the span tree shows exactly where to optimize.

**Explanation**

A from-scratch waterfall renderer - it takes a list of spans with start/duration/parent and prints ASCII bars scaled to the request width. Beyond the drawing, it computes the two numbers that matter for an LLM app: what fraction the model span dominates, and the real blank-screen wait a user feels (time to reach the model plus TTFT inside it).

**How the code works - step by step**
- **`spans`** - five tuples `(name, parent, start_ms, duration_ms)` describing one request's tree.
- **`TTFT_MS`** - first streamed token arrived 800ms into the model span.
- **`scale`** - maps the root duration onto 48 characters so bars are proportional.
- **Waterfall loop** - for each span, indents children, pads by start time, draws a `#` bar sized by duration, and prints its percentage of the request.
- **Reading block** - extracts `llm_generate`'s start and duration, reports it as ~95% of the request, and adds the reach-the-model time to TTFT for the true blank-screen wait.

**In one line:** scale each span to a bar, stack them into a waterfall, and read off that the model call dominates.

**What you'll see:** An ASCII waterfall for a 6130ms request - chat_request 100%, retrieve 3%, llm_generate 95%, tool_call and format_reply 1% each - followed by lines noting the model span is 95%, TTFT was 800ms, and the real wait before any text appeared was 230+800=1030ms.

## 4 - OpenTelemetry and the gen_ai conventions

How do you emit those spans without marrying one vendor for life? OpenTelemetry's `gen_ai` semantic conventions name the LLM attributes once - model, token usage, finish reason, TTFT - so a single instrumentation exports unchanged to any backend. This cell builds one such span and fans it out.

In [ ]:
# Instrument ONCE with OpenTelemetry's gen_ai semantic conventions, export ANYWHERE.
# The span is vendor-neutral: the same attributes render in Langfuse, Phoenix, or Datadog.
def gen_ai_span(model, in_tok, out_tok, ttft_ms, dur_ms, finish):
    # keys follow the OTel gen_ai.* convention (stabilizing in 2026)
    return {
        "name": "chat",
        "kind": "CLIENT",
        "gen_ai.operation.name": "chat",
        "gen_ai.system": "anthropic",
        "gen_ai.request.model": model,
        "gen_ai.usage.input_tokens": in_tok,
        "gen_ai.usage.output_tokens": out_tok,
        "gen_ai.response.finish_reasons": [finish],
        "gen_ai.server.time_to_first_token_ms": ttft_ms,
        "duration_ms": dur_ms,
    }

span = gen_ai_span("claude-sonnet-4-6", 1800, 240, 800, 5800, "end_turn")
print("One gen_ai span (created once, in your app):")
for k, v in span.items():
    print("    {:<38} {}".format(k, v))
print()
# The SAME span object is what every backend ingests - no re-instrumentation.
BACKENDS = ["Langfuse", "Arize Phoenix", "Datadog"]
print("Exported unchanged to each backend via OTLP:")
for b in BACKENDS:
    print("    {:<14} reads model={}, tokens={}+{}, ttft={}ms - same span, zero code changes".format(
        b, span["gen_ai.request.model"], span["gen_ai.usage.input_tokens"],
        span["gen_ai.usage.output_tokens"], span["gen_ai.server.time_to_first_token_ms"]))
print()
print("Wire to a vendor's proprietary SDK instead and you re-instrument every call to switch tools.")
print("OTel gen_ai conventions = instrument once, swap backends freely (no lock-in).")

# Output:
# One gen_ai span (created once, in your app):
#     name                                   chat
#     kind                                   CLIENT
#     gen_ai.operation.name                  chat
#     gen_ai.system                          anthropic
#     gen_ai.request.model                   claude-sonnet-4-6
#     gen_ai.usage.input_tokens              1800
#     gen_ai.usage.output_tokens             240
#     gen_ai.response.finish_reasons         ['end_turn']
#     gen_ai.server.time_to_first_token_ms   800
#     duration_ms                            5800
#
# Exported unchanged to each backend via OTLP:
#     Langfuse       reads model=claude-sonnet-4-6, tokens=1800+240, ttft=800ms - same span, zero code changes
#     Arize Phoenix  reads model=claude-sonnet-4-6, tokens=1800+240, ttft=800ms - same span, zero code changes
#     Datadog        reads model=claude-sonnet-4-6, tokens=1800+240, ttft=800ms - same span, zero code changes
#
# Wire to a vendor's proprietary SDK instead and you re-instrument every call to switch tools.
# OTel gen_ai conventions = instrument once, swap backends freely (no lock-in).

**Explanation**

A construct-once, export-anywhere demonstration - it builds a dict whose keys follow the OTel `gen_ai.*` naming standard, then prints that the identical object is ingested by three different backends. The idea is portability: standard attribute names mean zero re-instrumentation when you swap or add an observability tool.

**How the code works - step by step**
- **`gen_ai_span(...)`** - returns a dict tagged with the standard OTel keys: `gen_ai.operation.name`, `gen_ai.system`, `gen_ai.request.model`, input/output token usage, finish reasons, and TTFT.
- **`span = gen_ai_span(...)`** - builds one concrete chat span for a claude-sonnet-4-6 call.
- **First print loop** - dumps every attribute of the span (created once, in your app).
- **`BACKENDS`** - a list of three tools: Langfuse, Arize Phoenix, Datadog.
- **Export loop** - prints that each backend reads the same model/tokens/TTFT from the unchanged span with zero code changes.
- **Closing lines** - contrast this with a proprietary SDK that forces re-instrumentation to switch.

**In one line:** name the attributes the OTel way once, and every backend understands the same span for free.

**What you'll see:** One gen_ai span printed as nine attribute rows, then three lines showing Langfuse, Arize Phoenix, and Datadog each reading `model=claude-sonnet-4-6, tokens=1800+240, ttft=800ms` unchanged, closing on the instrument-once, no-lock-in takeaway.

## 5 - The golden signals

With spans flowing, what goes on the dashboard? SRE's classic golden signals - latency, traffic, errors, saturation - stay, and an LLM app adds TTFT, tokens, cost per request, cache hit rate, and a quality score. This cell computes all of them from a window of request records.

In [ ]:
# The golden signals for an LLM app, computed from a window of request records.
# Ops watches latency + errors; an AI app ALSO needs TTFT, tokens, cost, cache, and quality.
# (latency_ms, ttft_ms, ok, in_tok, out_tok, cache_hit, quality)
W = [
    (2100, 700, True,  1800, 240, False, 8.6),
    (2400, 820, True,  1500, 300, True,  8.1),
    (6100, 2100, True, 2000, 260, False, 6.9),
    (2200, 690, True,  1600, 210, True,  8.4),
    (2000, 640, False, 1700, 0,   False, 0.0),   # a failed request
    (2600, 900, True,  1900, 280, False, 8.0),
]
PRICE_IN, PRICE_OUT = 3.0 / 1_000_000, 15.0 / 1_000_000   # $/token (illustrative)

def pctl(values, p):
    s = sorted(values); k = max(0, int(round((p / 100.0) * (len(s) - 1))))
    return s[k]

ok = [r for r in W if r[2]]
lat = [r[0] for r in W]
cost = sum(r[3] * PRICE_IN + r[4] * PRICE_OUT for r in W)
print("Golden signals over a {}-request window:".format(len(W)))
print("  latency  p50 {}ms   p95 {}ms".format(pctl(lat, 50), pctl(lat, 95)))
print("  TTFT     p50 {}ms   p95 {}ms".format(pctl([r[1] for r in ok], 50), pctl([r[1] for r in ok], 95)))
print("  error rate       {:.0%}   ({} of {} failed)".format(1 - len(ok) / len(W), len(W) - len(ok), len(W)))
print("  throughput       {} requests in the window".format(len(W)))
print("  tokens           {} in / {} out".format(sum(r[3] for r in W), sum(r[4] for r in W)))
print("  cost             ${:.4f} total  (${:.4f} per request)".format(cost, cost / len(W)))
print("  cache hit rate   {:.0%}".format(sum(1 for r in W if r[5]) / len(W)))
print("  quality (mean)   {:.1f} / 10   (scored answers only)".format(sum(r[6] for r in ok) / len(ok)))
print()
print("Same signals, sliced by model or team, are your dashboard. p95 (not the mean) is what users feel;")
print("TTFT is what they SEE first; cost and quality are the two ops dashboards never show.")

# Output:
# Golden signals over a 6-request window:
#   latency  p50 2200ms   p95 6100ms
#   TTFT     p50 820ms   p95 2100ms
#   error rate       17%   (1 of 6 failed)
#   throughput       6 requests in the window
#   tokens           10500 in / 1290 out
#   cost             $0.0509 total  ($0.0085 per request)
#   cache hit rate   33%
#   quality (mean)   8.0 / 10   (scored answers only)
#
# Same signals, sliced by model or team, are your dashboard. p95 (not the mean) is what users feel;
# TTFT is what they SEE first; cost and quality are the two ops dashboards never show.

**Explanation**

A metrics-aggregation harness - it takes a window of request tuples and rolls them up into the full golden-signals panel. The teaching point rides in the percentile helper: reading p95 (the tail) instead of the mean is what surfaces the pain users actually feel, and cost/quality are the two signals a normal dashboard never computes.

**How the code works - step by step**
- **`W`** - six request tuples `(latency, ttft, ok, in_tok, out_tok, cache_hit, quality)`, one of them a failed request.
- **`PRICE_IN` / `PRICE_OUT`** - illustrative per-token dollar prices for the cost line.
- **`pctl(values, p)`** - sorts and index-selects a percentile (used for p50/p95).
- **`ok`** - filters to successful requests so quality and TTFT ignore the failure.
- **Print block** - reports latency p50/p95, TTFT p50/p95, error rate, throughput, total tokens, total and per-request cost, cache hit rate, and mean quality.
- **Closing lines** - note these same signals sliced by model or team are the actual dashboard.

**In one line:** roll a window of requests into the golden-signals panel, reading p95 (not the mean) for what users feel.

**What you'll see:** A golden-signals readout over a 6-request window: latency p50 2200/p95 6100ms, TTFT p50 820/p95 2100ms, error rate 17%, 10500 in/1290 out tokens, $0.0509 total ($0.0085/request), 33% cache hits, and mean quality 8.0/10.

## 6 - Alerting on SLOs and anomalies

Signals only help if the right one wakes the right person at the right time - and over-alerting is worse than none. This cell anchors alerts to an SLO with an error budget and enforces the key discipline: page on a sustained breach, log the self-healing blip.

In [ ]:
# Alert on SYMPTOMS against an SLO, and stay quiet on a blip. A brief spike is not an incident.
SLO = {"p95_ms": 3000, "error_budget": 0.005, "cost_ratio_max": 1.5, "quality_floor": 7.5}

def evaluate(window):
    fires = []
    if window["p95_ms"] > SLO["p95_ms"]:
        fires.append(("p95 latency", "{}ms > {}ms SLO".format(window["p95_ms"], SLO["p95_ms"])))
    if window["error_rate"] > SLO["error_budget"]:
        fires.append(("error budget", "{:.1%} > {:.1%} budget".format(window["error_rate"], SLO["error_budget"])))
    if window["cost_ratio"] > SLO["cost_ratio_max"]:
        fires.append(("cost anomaly", "{:.1f}x baseline > {:.1f}x".format(window["cost_ratio"], SLO["cost_ratio_max"])))
    if window["quality"] < SLO["quality_floor"]:
        fires.append(("quality drop", "{:.1f} < {:.1f} floor".format(window["quality"], SLO["quality_floor"])))
    return fires

# A rule only PAGES if it breaches for 3 consecutive windows (a blip must not wake anyone).
def page_or_quiet(history):
    breached = [len(evaluate(w)) > 0 for w in history]
    return all(breached[-3:]) and len(breached) >= 3

NORMAL  = {"p95_ms": 2400, "error_rate": 0.002, "cost_ratio": 1.1, "quality": 8.3}
BLIP    = {"p95_ms": 4200, "error_rate": 0.004, "cost_ratio": 1.2, "quality": 8.1}
INCIDENT= {"p95_ms": 5200, "error_rate": 0.030, "cost_ratio": 1.9, "quality": 6.4}

print("Three windows, evaluated against the SLO:")
for label, w in [("normal", NORMAL), ("brief blip", BLIP), ("real incident", INCIDENT)]:
    fires = evaluate(w)
    tag = "OK" if not fires else ", ".join(n for n, _ in fires)
    print("  {:<14} -> {}".format(label, tag if fires else "all signals within SLO"))
print()
print("But WHEN to page (need 3 consecutive bad windows, or a blip wakes on-call for nothing):")
print("  normal, normal, blip           -> " + ("PAGE" if page_or_quiet([NORMAL, NORMAL, BLIP]) else "stay quiet"))
print("  blip, incident, incident, incident -> " + ("PAGE" if page_or_quiet([BLIP, INCIDENT, INCIDENT, INCIDENT]) else "stay quiet"))
print()
print("Page on sustained SLO breaches, log the blips. Every false page trains on-call to ignore the next.")

# Output:
# Three windows, evaluated against the SLO:
#   normal         -> all signals within SLO
#   brief blip     -> p95 latency
#   real incident  -> p95 latency, error budget, cost anomaly, quality drop
#
# But WHEN to page (need 3 consecutive bad windows, or a blip wakes on-call for nothing):
#   normal, normal, blip           -> stay quiet
#   blip, incident, incident, incident -> PAGE
#
# Page on sustained SLO breaches, log the blips. Every false page trains on-call to ignore the next.

**Explanation**

A rule-engine harness with two layers - `evaluate` scores a single window against the SLO, and `page_or_quiet` decides whether a *history* of windows warrants paging. The core idea is the consecutive-breach requirement: it deliberately swallows a one-window blip so on-call is only paged for a sustained incident, the antidote to alert fatigue.

**How the code works - step by step**
- **`SLO`** - the objective: p95 ceiling, error budget, max cost ratio, quality floor.
- **`evaluate(window)`** - returns a list of fired rules (latency, error budget, cost anomaly, quality drop) for one window.
- **`page_or_quiet(history)`** - pages only if the last three windows all breached and at least three exist.
- **`NORMAL` / `BLIP` / `INCIDENT`** - three sample windows: clean, one-signal spike, and multi-signal failure.
- **First loop** - runs `evaluate` on each and prints which signals fired.
- **Paging checks** - runs `page_or_quiet` on a blip-ending history (stays quiet) and a sustained-incident history (pages).

**In one line:** score each window against the SLO, but only page when several bad windows stack up.

**What you'll see:** Three windows evaluated - normal is clean, the blip trips p95 only, the incident trips all four rules - then two paging decisions: `normal, normal, blip -> stay quiet` and `blip, incident, incident, incident -> PAGE`, closing on the alert-fatigue warning.

## 7 - Online eval, drift, and PII

The last piece is the signal that makes this an *AI* observability stack: score a sample of live traffic to catch slow quality drift days before any error fires - and redact PII before it ever enters a span. This cell runs a rolling-quality drift check and a regex redactor.

In [ ]:
# Online eval: score a sample of LIVE traffic and watch the trend for DRIFT.
# Separately: redact PII BEFORE anything enters a trace.
import re

# A rolling quality score over several days (an LLM judge on sampled live replies, 0-10).
daily_quality = [8.4, 8.3, 8.1, 7.6, 6.8, 6.2]
FLOOR = 7.0
window = 3   # alert if the rolling mean of the last 3 days falls below the floor

print("Online eval - rolling answer quality on sampled live traffic:")
for i in range(len(daily_quality)):
    lo = max(0, i - window + 1)
    roll = sum(daily_quality[lo:i + 1]) / (i - lo + 1)
    flag = "  <- DRIFT ALERT (below {} floor)".format(FLOOR) if roll < FLOOR else ""
    print("  day {}: score {:.1f}   rolling({}) {:.2f}{}".format(i + 1, daily_quality[i], window, roll, flag))
print("  quality drifted down for days before any error fired - only an eval loop catches this.")
print()

# PII must be redacted BEFORE the prompt/response is written into a span.
def redact(text):
    text = re.sub(r"[\w.+-]+@[\w-]+\.[\w.-]+", "[EMAIL]", text)
    text = re.sub(r"\b\d{3}[-.\s]?\d{3}[-.\s]?\d{4}\b", "[PHONE]", text)
    return text

raw = "Refund to priya.n@example.com or call 555-018-2234 about order 7781"
print("PII redaction before capture:")
print("  raw prompt     : " + raw)
print("  stored in trace: " + redact(raw))
print("  the email and phone never touch your telemetry backend.")

# Output:
# Online eval - rolling answer quality on sampled live traffic:
#   day 1: score 8.4   rolling(3) 8.40
#   day 2: score 8.3   rolling(3) 8.35
#   day 3: score 8.1   rolling(3) 8.27
#   day 4: score 7.6   rolling(3) 8.00
#   day 5: score 6.8   rolling(3) 7.50
#   day 6: score 6.2   rolling(3) 6.87  <- DRIFT ALERT (below 7.0 floor)
#   quality drifted down for days before any error fired - only an eval loop catches this.
#
# PII redaction before capture:
#   raw prompt     : Refund to priya.n@example.com or call 555-018-2234 about order 7781
#   stored in trace: Refund to [EMAIL] or call [PHONE] about order 7781
#   the email and phone never touch your telemetry backend.

**Explanation**

Two safety mechanisms in one cell - a rolling-average drift detector and a PII scrubber. The drift loop is the online-eval loop (the CI eval gate from 12.6, now continuous) that catches a creeping regression the error rate never shows; the `redact` function enforces the hard rule that emails and phone numbers must be masked *before* text is written to a trace, not after.

**How the code works - step by step**
- **`daily_quality`** - six days of sampled LLM-judge scores, trending downward.
- **`FLOOR` / `window`** - the alert floor and the 3-day rolling window.
- **Drift loop** - computes each day's trailing rolling mean and flags a DRIFT ALERT once it drops below the floor.
- **`redact(text)`** - two regex subs replacing email addresses with `[EMAIL]` and phone numbers with `[PHONE]`.
- **PII block** - runs `redact` on a raw prompt and prints raw-vs-stored so you see the sensitive fields never reach the backend.

**In one line:** watch the rolling quality mean cross a floor to catch drift, and scrub PII at the door before capture.

**What you'll see:** Six days of rolling quality (8.40 down to 6.87) with a DRIFT ALERT firing on day 6 below the 7.0 floor, then a redaction demo turning `priya.n@example.com` and `555-018-2234` into `[EMAIL]` and `[PHONE]` in the stored trace.

Together these seven models are a complete AI-observability stack in miniature: reject the green-dashboard illusion (Cell 1), capture behavior through logs/metrics/traces plus evals (Cells 2-3), emit spans in a vendor-neutral gen_ai format (Cell 4), put the golden signals on a dashboard (Cell 5), alert only on sustained SLO breaches (Cell 6), and watch live quality for drift while scrubbing PII at the door (Cell 7). Every cell runs with no backend and no API key - the one exception is the illustrative OTel+Langfuse function, shown for reference. Next: Lesson 12.9 uses this telemetry to diagnose an outage, Lesson 14.4 builds the eval set your online-eval loop scores against, and Module 13 cuts the cost this dashboard surfaces.